# 🏷️ Elasticidad de Precio (Regresión Log-Log)
> **Caso de negocio:** Retail multicategoría · Pricing & Promociones
> **Framework:** CRISP-DM · Digital Marketing Analytics — UPC
> **Autor:** Miguel Salazar · Attach Group

---

## Contexto del problema

Una cadena de retail multicategoría (Electrónica, Ropa, Hogar, Deportes) ajusta precios y promociones **de forma intuitiva**, sin medir cómo responde la demanda a cambios de precio en cada categoría.

**Objetivo:** Estimar la elasticidad precio-demanda global y por categoría, para identificar en qué categorías subir precios afecta poco la demanda (inelásticas) y en cuáles un descuento genera mucho volumen adicional (elásticas).

**KPIs de éxito:**
- R² >= 0.5 en el modelo log-log
- Elasticidad por categoría con signo negativo y coherente (bienes normales)
- Recomendación de pricing diferenciada por categoría

**Algoritmo:** Regresión lineal log-log (`log(unidades) ~ log(precio) + promoción`). El coeficiente de `log(precio)` es directamente la elasticidad precio-demanda.

## 📦 Fase 0 — Setup

In [ ]:
!pip install -q plotly

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Setup completo ✓')

## 1️⃣ Fase 1 — Business Understanding

In [ ]:
PROBLEMA = {
    'tipo': 'Regresión supervisada (log-log)',
    'target': 'log_unidades (log de unidades vendidas por semana/categoría)',
    'unidad_analisis': 'semana x categoría',
    'features': [
        'log_precio  → log del precio normalizado dentro de cada categoría',
        'promocion   → 1 si la semana tuvo promoción activa, 0 si no',
    ],
    'criterios_aceptacion': {
        'R2': '>= 0.50',
        'Elasticidad': '< 0 en todas las categorías (consistente con bienes normales)',
    }
}

for k, v in PROBLEMA.items():
    if isinstance(v, list):
        print(f'\n{k}:')
        for item in v: print(f'  {item}')
    elif isinstance(v, dict):
        print(f'\n{k}:')
        for mk, mv in v.items(): print(f'  {mk}: {mv}')
    else:
        print(f'{k}: {v}')

## 2️⃣ Fase 2 — Data Understanding

In [ ]:
# Generación de datos sintéticos con estructura realista
# En producción: reemplazar con datos de ventas semanales por SKU/categoría (BigQuery)
N = 800

cats = ['Electronica', 'Ropa', 'Hogar', 'Deportes']
base_prices = {'Electronica': 350, 'Ropa': 120, 'Hogar': 85, 'Deportes': 200}
# Elasticidades "reales" del DGP — Electrónica es la más sensible al precio
betas = {'Electronica': -1.8, 'Ropa': -1.2, 'Hogar': -0.8, 'Deportes': -1.4}

rows = []
for i in range(N):
    cat = np.random.choice(cats)
    promo = int(np.random.random() > 0.8)
    bp = base_prices[cat]
    precio = round(bp * (0.65 + np.random.random() * 0.7) / 5) * 5
    beta = betas[cat]
    dem_base = 1000
    units = round(dem_base * (precio / bp) ** beta * (1 + promo * 0.15)
                   * (0.9 + np.random.random() * 0.2))
    rows.append({'semana': f'S{i+1}', 'precio': precio,
                  'unidades': max(units, 10), 'categoria': cat, 'promocion': promo})

df = pd.DataFrame(rows)

print(f'Dataset: {df.shape}')
print(f'\nObservaciones por categoría:')
print(df['categoria'].value_counts())
df.head()

In [ ]:
# Distribución de precio y unidades por categoría
fig = make_subplots(rows=1, cols=2, subplot_titles=['Precio por categoría', 'Unidades vendidas por categoría'])

colors = {'Electronica': '#1a4c8c', 'Ropa': '#c0392b', 'Hogar': '#0d9488', 'Deportes': '#d97706'}

for cat in cats:
    sub = df[df['categoria'] == cat]
    fig.add_trace(go.Box(y=sub['precio'], name=cat, marker_color=colors[cat], showlegend=False), row=1, col=1)
    fig.add_trace(go.Box(y=sub['unidades'], name=cat, marker_color=colors[cat], showlegend=False), row=1, col=2)

fig.update_layout(height=400, title='Distribución de precio y unidades vendidas',
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Relación precio vs unidades, por categoría (escala original)
fig = go.Figure()
for cat in cats:
    sub = df[df['categoria'] == cat]
    fig.add_trace(go.Scatter(x=sub['precio'], y=sub['unidades'], mode='markers',
                              name=cat, marker_color=colors[cat], opacity=0.6))

fig.update_layout(title='Demanda vs Precio, por categoría',
                  xaxis_title='Precio (S/.)', yaxis_title='Unidades vendidas',
                  height=420, plot_bgcolor='white', paper_bgcolor='white')
fig.update_xaxes(gridcolor='#f0f0f0')
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()

## 3️⃣ Fase 3 — Data Preparation

In [ ]:
# Normalizar el precio dentro de cada categoría (elimina el efecto del nivel de precio entre categorías)
# y aplicar transformación log-log
df['precio_norm']  = df.groupby('categoria')['precio'].transform(lambda x: x / x.mean())
df['log_precio']   = np.log(df['precio_norm'])
df['log_unidades'] = np.log(df['unidades'])

FEATURES = ['log_precio', 'promocion']
TARGET   = 'log_unidades'

X = df[FEATURES].fillna(0)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
df[['categoria', 'precio', 'precio_norm', 'log_precio', 'unidades', 'log_unidades', 'promocion']].head()

## 4️⃣ Fase 4 — Modeling

In [ ]:
# Regresión log-log global: el coeficiente de log_precio es la elasticidad precio-demanda
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

elasticity = model.coef_[0]
coef_promo = model.coef_[1]

print(f'Elasticidad precio-demanda global: {elasticity:.3f}')
print(f'Efecto de la promoción (log-puntos): {coef_promo:.3f}  → ~{(np.exp(coef_promo)-1)*100:.1f}% en unidades')

In [ ]:
# Elasticidad por categoría: una regresión log-log independiente por cada categoría
elast_by_cat = {}
for cat in cats:
    sub = df[df['categoria'] == cat].copy()
    sub['log_p'] = np.log(sub['precio'])
    sub['log_q'] = np.log(sub['unidades'])
    m = LinearRegression()
    m.fit(sub[['log_p']], sub['log_q'])
    elast_by_cat[cat] = round(m.coef_[0], 3)

elast_df = pd.DataFrame({
    'categoria': list(elast_by_cat.keys()),
    'elasticidad': list(elast_by_cat.values())
}).sort_values('elasticidad')

elast_df['tipo'] = np.where(elast_df['elasticidad'].abs() > 1, 'Elástica', 'Inelástica')
display(elast_df)

## 5️⃣ Fase 5 — Evaluation

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

metrics = {'R2': r2, 'RMSE (log)': rmse, 'Elasticidad global': elasticity}
print('=== MÉTRICAS DEL MODELO GLOBAL (test set) ===')
for k, v in metrics.items():
    print(f'  {k:20s}: {v:.4f}')

estado = '✅ APROBADO' if r2 >= 0.50 else '❌ NECESITA MEJORA (umbral >= 0.50)'
print(f'\nCriterio R2 >= 0.50: {estado}')

In [ ]:
# Curva de demanda vs precio (escala original), con la elasticidad global anotada
fig = go.Figure()
for cat in cats:
    sub = df[df['categoria'] == cat]
    fig.add_trace(go.Scatter(x=sub['precio'], y=sub['unidades'], mode='markers',
                              name=cat, marker_color=colors[cat], opacity=0.6))

fig.update_layout(title=f'Demanda vs Precio (elasticidad global: {elasticity:.2f})',
                  xaxis_title='Precio (S/.)', yaxis_title='Unidades vendidas',
                  height=420, plot_bgcolor='white', paper_bgcolor='white')
fig.update_xaxes(gridcolor='#f0f0f0')
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()

In [ ]:
# Elasticidad por categoría
fig = go.Figure(go.Bar(
    x=elast_df['elasticidad'], y=elast_df['categoria'], orientation='h',
    marker_color=[colors[c] for c in elast_df['categoria']],
    text=elast_df['elasticidad'].map('{:.2f}'.format), textposition='outside'
))
fig.add_vline(x=-1, line_dash='dash', line_color='gray',
              annotation_text='Umbral elástico/inelástico (-1)')
fig.update_layout(title='Elasticidad precio-demanda por categoría',
                  height=350, plot_bgcolor='white', paper_bgcolor='white',
                  xaxis=dict(title='Elasticidad', gridcolor='#f0f0f0'),
                  yaxis=dict(showgrid=False))
fig.show()

## 6️⃣ Fase 6 — Deployment

In [ ]:
# Simulación de escenario: ¿qué pasa con ingresos si subimos precios +10% por categoría?
delta_precio = 0.10  # +10%

sim = elast_df.copy()
sim['delta_unidades_%'] = (((1 + delta_precio) ** sim['elasticidad']) - 1) * 100
sim['delta_ingresos_%'] = (((1 + delta_precio) ** (1 + sim['elasticidad'])) - 1) * 100
sim['recomendacion'] = np.where(
    sim['delta_ingresos_%'] > 0,
    '✅ Subir precio aumenta ingresos',
    '⚠️ Subir precio reduce ingresos — evaluar promoción en su lugar'
)

display(sim[['categoria', 'elasticidad', 'delta_unidades_%', 'delta_ingresos_%', 'recomendacion']].round(2))

In [ ]:
# Guardar outputs
sim.to_csv('elasticidad_por_categoria.csv', index=False)
print('Archivo elasticidad_por_categoria.csv guardado ✓')

import joblib
joblib.dump({'model': model, 'features': FEATURES, 'elasticity_by_cat': elast_by_cat},
            'modelo_elasticidad.pkl')
print('Modelo modelo_elasticidad.pkl guardado ✓')

In [ ]:
# Resumen ejecutivo
cat_inelastica = elast_df.loc[elast_df['elasticidad'].abs().idxmin()]
cat_elastica   = elast_df.loc[elast_df['elasticidad'].abs().idxmax()]

print('=== RESUMEN EJECUTIVO ===')
print(f'Elasticidad global estimada:        {elasticity:.2f}')
print(f'Categoría más inelástica:           {cat_inelastica["categoria"]} (e={cat_inelastica["elasticidad"]:.2f}) → margen para subir precio')
print(f'Categoría más elástica:             {cat_elastica["categoria"]} (e={cat_elastica["elasticidad"]:.2f}) → priorizar promociones para volumen')
print(f'Efecto promoción (en unidades):     ~{(np.exp(coef_promo)-1)*100:.1f}%')
print(f'\nArquitectura de producción:')
print('  Ventas semanales por SKU → BigQuery → recalculo mensual de elasticidad → motor de pricing')